# 3.2 Arquitectura del Data Warehouse: El Esquema de Estrella (Banca)
Vamos a simular un Data Warehouse en memoria. Tenemos una **Tabla de Hechos** (las transacciones crudas) rodeada de **Tablas de Dimensión** (el contexto de la transacción). Nuestro objetivo de KDD es crear una matriz plana para alimentar a un modelo de riesgo.

In [1]:
import pandas as pd

print("--- 1. TABLAS DE DIMENSIÓN (El Contexto) ---")
# Dimensión Cliente (Desnormalizada: el segmento ya viene escrito, no es otro ID)
dim_clientes = pd.DataFrame({
    'id_cliente': [101, 102, 103, 104],
    'estado': ['BC', 'CDMX', 'BC', 'Jalisco'],
    'segmento': ['Universitario', 'Premium', 'Universitario', 'Standard']
})

display(dim_clientes)

# Dimensión Tiempo
dim_tiempo = pd.DataFrame({
    'id_tiempo': [20260914, 20260915],
    'dia_semana': ['Lunes', 'Martes'],
    'es_quincena': [False, True]
})
display(dim_tiempo)

print("\n--- 2. TABLA DE HECHOS (La Métrica) ---")
# La tabla de hechos solo une IDs numéricos y la métrica (Monto)
fact_transacciones = pd.DataFrame({
    'id_transaccion': [1, 2, 3, 4, 5],
    'id_cliente': [101, 102, 101, 104, 103],
    'id_tiempo': [20260914, 20260914, 20260915, 20260915, 20260915],
    'monto_mxn': [150.50, 4500.00, 85.00, 1200.00, 300.00]
})


display(fact_transacciones)

--- 1. TABLAS DE DIMENSIÓN (El Contexto) ---


,id_cliente,estado,segmento
0,101,BC,Universitario
1,102,CDMX,Premium
2,103,BC,Universitario
3,104,Jalisco,Standard


,id_tiempo,dia_semana,es_quincena
0,20260914,Lunes,False
1,20260915,Martes,True



--- 2. TABLA DE HECHOS (La Métrica) ---


,id_transaccion,id_cliente,id_tiempo,monto_mxn
0,1,101,20260914,150.5
1,2,102,20260914,4500.0
2,3,101,20260915,85.0
3,4,104,20260915,1200.0
4,5,103,20260915,300.0


### Ejecutando la Tubería ETL en Memoria
Para que el modelo de Machine Learning entienda esto, debemos pasar por las tres fases de la arquitectura de datos.

In [2]:
print("--- INICIANDO PIPELINE ETL ---")

# EXTRACT & TRANSFORM: Cruzamos los hechos con las dimensiones (Desnormalización masiva)
matriz_kdd = pd.merge(fact_transacciones, dim_clientes, on='id_cliente', how='inner')
matriz_kdd = pd.merge(matriz_kdd, dim_tiempo, on='id_tiempo', how='inner')

# TRANSFORM: Purgamos llaves primarias operativas que causan ruido matemático
matriz_kdd = matriz_kdd.drop(columns=['id_transaccion', 'id_cliente', 'id_tiempo'])

# LOAD: La matriz final se "carga" en la variable de memoria lista para la IA
print("--- MATRIZ MULTIDIMENSIONAL (LOAD COMPLETO) ---")
display(matriz_kdd)

--- INICIANDO PIPELINE ETL ---
--- MATRIZ MULTIDIMENSIONAL (LOAD COMPLETO) ---


,monto_mxn,estado,segmento,dia_semana,es_quincena
0,150.5,BC,Universitario,Lunes,False
1,4500.0,CDMX,Premium,Lunes,False
2,85.0,BC,Universitario,Martes,True
3,1200.0,Jalisco,Standard,Martes,True
4,300.0,BC,Universitario,Martes,True


---
### 🛠️ Zona de Experimentación y Reto de Código

En los Almacenes de Datos, la operación analítica clásica se llama **Roll-up** (Acumular o Resumir datos subiendo de granularidad). 

**Reto 1: Código Analítico**
Usa Pandas para responder a la pregunta que tiró el servidor de producción: *¿Cuál es el monto total de dinero transaccionado por cada Segmento de Cliente?*
Escribe el código usando la función `.groupby()` sobre tu `matriz_kdd`.

In [3]:
resultado_rollup = matriz_kdd.groupby('segmento')['monto_mxn'].sum()
print(resultado_rollup)

segmento
Premium          4500.0
Standard         1200.0
Universitario     535.5
Name: monto_mxn, dtype: float64


**Reto 2: Arquitectura en Papel (Responde aquí mismo)**
Observa nuestra tabla `dim_clientes`. En una base de datos relacional estricta (OLTP), el segmento "Universitario" sería un número (un ID_Segmento) que apuntaría a otra tabla de catálogo. 
En el Almacén de Datos (OLAP) decidimos escribir "Universitario" directamente en texto, repitiéndolo por cada cliente. 
Como ingeniero de software: ¿Por qué estamos violando las reglas de normalización a propósito en esta arquitectura? ¿Qué ganamos al aceptar esa redundancia?

> **Tu respuesta:** Porque un JOIN cuesta más que el disco. Repetir el texto deja la consulta en una sola tabla, y OLAP casi no escribe, que es justo lo que la normalización protege.

---
# Caso: Data Warehouse para Clínicas Dentales

Imagina que fuiste contratado para diseñar el Data Warehouse analítico de una plataforma web (basada en Vue.js) para Clínicas Dentales. La aplicación de producción gestiona expedientes de pacientes y los resultados de un algoritmo de IA que segmenta bacterias en las radiografías.

El director médico solicita un análisis mensual.

> *"¿Qué rango de edades y de qué sucursal muestran el mayor porcentaje de bacterias durante las visitas de fin de semana?"*

**Tabla en crudo:**

| ID_Visita | Paciente | Edad_Paciente | Sucursal_Atencion | Ciudad | Fecha_Consulta | Dia_Semana | Porcentaje_Bacterias | Tiempo_Inferencia_IA_Seg | Mascara_IA_Base64 |
|---|---|---|---|---|---|---|---|---|---|
| 8001 | Raúl Soto | 45 | Clínica Centro | Ensenada | 2026-10-03 | Sábado | 68.5 | 1.2 | iVBORw0KGgoAAAANSU... |
| 8002 | Sofía Lira | 12 | Clínica Valle | Tijuana | 2026-10-03 | Sábado | 15.0 | 0.8 | iVBORw0KGgoAAAANSU... |
| 8003 | Raúl Soto | 45 | Clínica Centro | Ensenada | 2026-10-04 | Domingo | 65.2 | 1.1 | iVBORw0KGgoAAAANSU... |
| 8004 | Ana Cruz | 68 | Clínica Sur | Ensenada | 2026-10-06 | Martes | 82.1 | 2.5 | iVBORw0KGgoAAAANSU... |
| 8005 | Luis Rey | 25 | Clínica Valle | Tijuana | 2026-10-06 | Martes | 5.4 | 0.9 | iVBORw0KGgoAAAANSU... |

**Diseña el esquema considerando lo siguiente:**

1. Define tu Tabla de Hechos (Fact_Segmentaciones_Dentales). ¿Qué métricas numéricas puras irán aquí? y ¿cuáles serán sus llaves foráneas?
2. Define al menos 3 Tablas de Dimensión. Nómbralas y detalla qué atributos descriptivos guardarás en cada una para resolver la pregunta del director.
3. Pregunta sobre ETL. Durante el proceso de Extract y Transform diario, descubres que la aplicación web transaccional guarda la máscara visual de la radiografía en base64 (una cadena de texto de 2 MB por fila). ¿Por qué sería un error catastrófico cargar (Load) ese string gigantesco en la Tabla de Hechos y cómo lo resolverías en tu tubería de datos?

---
**1. Define tu Tabla de Hechos (Fact_Segmentaciones_Dentales). ¿Qué métricas numéricas puras irán aquí? y ¿cuáles serán sus llaves foráneas?**

> **Tu respuesta:** Las métricas son porcentaje_bacterias y tiempo_inferencia_ia_seg. Las llaves foráneas son id_paciente, id_sucursal e id_tiempo, más id_visita como llave degenerada.

---
**2. Define al menos 3 Tablas de Dimensión. Nómbralas y detalla qué atributos descriptivos guardarás en cada una para resolver la pregunta del director.**

> **Tu respuesta:**
> 
> Dim_Paciente con id_paciente, nombre, edad y rango_edad precalculado con pd.cut.
> 
> Dim_Sucursal con id_sucursal, nombre_sucursal y ciudad.
> 
> Dim_Tiempo con id_tiempo, fecha, dia_semana, es_fin_semana, mes y anio.

---
**3. Pregunta sobre ETL. Durante el proceso de Extract y Transform diario, descubres que la aplicación web transaccional guarda la máscara visual de la radiografía en base64 (una cadena de texto de 2 MB por fila). ¿Por qué sería un error catastrófico cargar (Load) ese string gigantesco en la Tabla de Hechos y cómo lo resolverías en tu tubería de datos?**

> **Tu respuesta:** La tabla de hechos es la que todo query escanea. Con 2 MB por fila, un millón de visitas son 2 TB leídos para sacar un promedio de 8 bytes, y una máscara ni siquiera se puede agregar. Se guarda la imagen en object storage y en la tabla de hechos solo queda la ruta.